#Prediction of the Winner of the ATP Finals Tournament
In the following notebook, I implement the full algorithmic pipeline used to estimate the probability that each player wins the ATP Finals tournament. Starting from historical ATP match data, I compute pre-tournament Elo ratings, transform them into head-to-head win probabilities, and then simulate the tournament many times through Monte Carlo. The final part of the notebook evaluates the predictive performance of the model through historical backtesting on the ATP Finals editions from 2000 to 2024.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
from collections import Counter

##Data Setup and Preprocessing
In this preliminary part of the notebook, we prepare the dataset that will be used in the next steps of the project. We make the code portable and easy to run on different devices by defining the project paths clearly, and we dynamically download the match files from Jeff Sackmann’s GitHub repository. Finally, we perform a basic cleaning step on the merged dataset. In particular, we keep the essential information needed for the following analysis and save the final dataset into a single file, so that the rest of the project can work with one unified and consistent dataset.

In [3]:
PROJECT_DIR = Path.cwd()
DATA_DIR    = PROJECT_DIR / "data"
RAW_DIR     = DATA_DIR / "raw"
PROC_DIR    = DATA_DIR / "processed"

for d in (RAW_DIR, PROC_DIR):
    d.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2000, 2025))

#Git Hub Jeff Sackmann
SACKMANN_BASE = "https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master"

def download_if_missing(url: str, out_path: Path) -> bool:
    if out_path.exists() and out_path.stat().st_size > 0:
        return False
    out_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, out_path.as_posix())
    return True

downloaded = 0
for y in YEARS:
    fname = f"atp_matches_{y}.csv"
    url   = f"{SACKMANN_BASE}/{fname}"
    out   = RAW_DIR / "sackmann" / fname
    if download_if_missing(url, out):
        downloaded += 1

In [4]:
#DOWNLOAD DI TUTTI I MATCH DAL 2000 AL 2024 DISPONIBILI SUL GIT HUB DI JEFF SACKMANN
sackmann_dir = RAW_DIR / "sackmann"
files = sorted(sackmann_dir.glob("atp_matches_*.csv"))

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["source_file"] = f.name
    dfs.append(df)

matches_all = pd.concat(dfs, ignore_index=True)
display(matches_all)

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,source_file
0,2000-301,Auckland,Hard,32,A,20000110,1,103163,1.0,NaN,...,39.0,29.0,17.0,4.0,7.0,11.0,1612.0,63.0,595.0,atp_matches_2000.csv
1,2000-301,Auckland,Hard,32,A,20000110,2,102607,NaN,Q,...,25.0,18.0,12.0,3.0,6.0,211.0,157.0,49.0,723.0,atp_matches_2000.csv
2,2000-301,Auckland,Hard,32,A,20000110,3,103252,NaN,NaN,...,20.0,7.0,8.0,7.0,11.0,48.0,726.0,59.0,649.0,atp_matches_2000.csv
3,2000-301,Auckland,Hard,32,A,20000110,4,103507,7.0,NaN,...,29.0,14.0,10.0,6.0,8.0,45.0,768.0,61.0,616.0,atp_matches_2000.csv
4,2000-301,Auckland,Hard,32,A,20000110,5,102103,NaN,Q,...,34.0,18.0,12.0,5.0,9.0,167.0,219.0,34.0,873.0,atp_matches_2000.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74901,2024-M-DC-2024-WG2-PO-URU-MDA-01,Davis Cup WG2 PO: URU vs MDA,Clay,4,D,20240203,5,212051,NaN,NaN,...,17.0,7.0,6.0,8.0,14.0,1109.0,8.0,740.0,34.0,atp_matches_2024.csv
74902,2024-M-DC-2024-WG2-PO-VIE-RSA-01,Davis Cup WG2 PO: VIE vs RSA,Hard,4,D,20240202,1,122533,NaN,NaN,...,25.0,6.0,9.0,1.0,4.0,554.0,67.0,748.0,32.0,atp_matches_2024.csv
74903,2024-M-DC-2024-WG2-PO-VIE-RSA-01,Davis Cup WG2 PO: VIE vs RSA,Hard,4,D,20240202,2,144748,NaN,NaN,...,25.0,7.0,11.0,5.0,12.0,416.0,109.0,NaN,NaN,atp_matches_2024.csv
74904,2024-M-DC-2024-WG2-PO-VIE-RSA-01,Davis Cup WG2 PO: VIE vs RSA,Hard,4,D,20240202,4,122533,NaN,NaN,...,32.0,17.0,14.0,5.0,9.0,554.0,67.0,416.0,109.0,atp_matches_2024.csv


In [5]:
#DATA CLEANING

# tourney_date in datetime (YYYYMMDD)
matches_all["tourney_date"] = pd.to_datetime(matches_all["tourney_date"], format="%Y%m%d", errors="coerce")

# Droppiamo righe con nomi e date: Nan
matches_all = matches_all.dropna(subset=["tourney_date", "winner_name", "loser_name"])

display(matches_all[["tourney_date","tourney_name","round","surface","winner_name","loser_name"]].head())

,tourney_date,tourney_name,round,surface,winner_name,loser_name
0,2000-01-10,Auckland,R32,Hard,Tommy Haas,Jeff Tarango
1,2000-01-10,Auckland,R32,Hard,Juan Balcells,Franco Squillari
2,2000-01-10,Auckland,R32,Hard,Alberto Martin,Alberto Berasategui
3,2000-01-10,Auckland,R32,Hard,Juan Carlos Ferrero,Roger Federer
4,2000-01-10,Auckland,R32,Hard,Michael Sell,Nicolas Escude


In [6]:
# SALVIAMO IL DATASET UNICO CON TUTTI I MATCH
out_path = PROC_DIR / "matches_all_2000_2024.csv"
matches_all.to_csv(out_path, index=False)

print("✅ Saved:", out_path)

✅ Saved: /content/data/processed/matches_all_2000_2024.csv


In the next blocks, I identify the ATP Finals event of each year within the full match dataset and extract its starting date and tournament ID. These pieces of information are needed to define, for every edition, the reference time window used to train the Elo ratings before the tournament. I then introduce simple helper functions to retrieve the Finals matches and the corresponding training matches for any selected year.

In [7]:
# DAL DATASET DI TUTTI I MATCH DOBBIAMO FILTRARE LE PARTITE DELLE FINALS DI OGNI ANNO PER POI STABILIRE QUALE FINESTRA DI TEMPO CONSIDERARE NEL CALCOLO DEL RANKING ELO

finals_mask = (
    matches_all["tourney_name"].str.contains("ATP Finals|Tour Finals|Masters Cup", case=False, na=False)
    & ~matches_all["tourney_name"].str.contains("Next Gen|NextGen", case=False, na=False)
)

finals_events = (
    matches_all.loc[finals_mask, ["tourney_id", "tourney_name", "tourney_date"]]
    .dropna(subset=["tourney_id", "tourney_date"])
    .drop_duplicates()
    .copy()
)

finals_events["year"] = finals_events["tourney_date"].dt.year

finals_by_year = (
    finals_events.sort_values(["year", "tourney_date"])
    .groupby("year")
    .tail(1)
    .set_index("year")
    .sort_index()
)

print("Finals years found:", finals_by_year.index.min(), "→", finals_by_year.index.max())
display(finals_by_year)

Finals years found: 2000 → 2024


,tourney_id,tourney_name,tourney_date
year,,,
2000,2000-605,Masters Cup,2000-11-27
2001,2001-605,Masters Cup,2001-11-12
2002,2002-605,Masters Cup,2002-11-11
2003,2003-605,Masters Cup,2003-11-10
2004,2004-605,Masters Cup,2004-11-15
2005,2005-605,Masters Cup,2005-11-14
2006,2006-605,Masters Cup,2006-11-13
2007,2007-605,Masters Cup,2007-11-12
2008,2008-605,Masters Cup,2008-11-09


In [8]:
#creiamo funzioni per estrarre date per finestra dove trainare l'ELO

def get_finals_start(year: int) -> pd.Timestamp:
    return finals_by_year.loc[year, "tourney_date"]

def get_finals_id(year: int) -> str:
    return finals_by_year.loc[year, "tourney_id"]

#la finestra di training la scegliamo partendo dalla data delle finals dell anno X come punto di fine
#e selezionando come data di inizio la fine delle finals dell'anno X-1 (data inizio anno (X-1) + 9 giorni )
def get_training_window(year: int):
    end = get_finals_start(year) - pd.Timedelta(days=1)
    if (year - 1) in finals_by_year.index:
        start = get_finals_start(year - 1) + pd.Timedelta(days=9)
    else:
        start = get_finals_start(year) - pd.Timedelta(days=340)
    return start, end

#funzione per estrarre le partite di training
def get_train_matches(matches: pd.DataFrame, year: int) -> pd.DataFrame:
    start, end = get_training_window(year)
    m = matches[(matches["tourney_date"] >= start) & (matches["tourney_date"] <= end)].copy()
    return m

#funzioni per estrarre solo partite delle finals
def get_finals_matches(matches: pd.DataFrame, year: int) -> pd.DataFrame:
    tid = get_finals_id(year)
    return matches[matches["tourney_id"] == tid].copy()

## RESOLUTION ALGORITHM

The core of the notebook is the resolution algorithm used to estimate the title-winning probability of each ATP Finals entrant. For a given year, the procedure first computes the pre-tournament Elo ratings of the 8 qualified players, then transforms these ratings into pairwise head-to-head win probabilities, and finally propagates these probabilities through the tournament structure by Monte Carlo simulation.

### Elo

In this block, I implement the weighted Elo procedure used to estimate player strength before the ATP Finals of a given year. In particular, I define the two main functions used to train and compute the pre-tournament Elo ranking, which will later be reused both in the toy example and in the full simulation. Starting from the matches in the selected training window, the code processes results sequentially and updates the ratings of winners and losers according to the standard Elo formula, enriched with optional adjustments for tournament level, surface, and time decay. At the end of the procedure, I extract both the complete rating dictionary and the ratings restricted to the 8 entrants of the considered edition.

In [9]:
# ELO PRE FINALS

# Parametri Elo:
R0 = 1500.0
K  = 32.0
S  = 400.0
HALFLIFE_DAYS = 180
LEVEL_WEIGHTS = {"G": 1.5, "M": 1.25, "A": 1.0, "D": 1.0 }
SURFACE_WEIGHTS = {"Hard": 1.0, "Clay": 0.7, "Grass": 0.8}
round_order = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]


#FUNZIONE PER PROBABILITA DI VITTORIA nella partita j vs i.
def elo_p(E_i: float, E_j: float, s: float = 400.0) -> float:
    return 1.0 / (1.0 + 10.0 ** ((E_j - E_i) / s))

#Funzioni per estrarre livello e superficie
def w_level(level: str) -> float:
    return float(LEVEL_WEIGHTS.get(level, 1.0))

def w_surface(surface: str) -> float:
    return float(SURFACE_WEIGHTS.get(surface, 1.0))


#FUNZIONE CHE AGGIORNA ELO per Winner e Loser per ogni match in df
def train_elo(df: pd.DataFrame, # df dovra avere colonne: date, winner, loser, level, surface, round
              finals_date: pd.Timestamp,
              R0: float = 1500.0,
              K: float = 32.0,
              S: float = 400.0,
              halflife_days: float = 180,
              use_decay: bool = True,
              use_level_weights: bool = True,
              use_surface_weights: bool = True) -> dict:

    R = {}
    inv_h = 1.0 / float(halflife_days) #lo calcoliamo una sola volta fuori dal for

    # Ordiniamo cronologicamente, per nome del torneo e per round (dentro al torneo)
    df["round"] = pd.Categorical(df["round"], categories=round_order, ordered=True)
    df = df.sort_values(["date", "tourney_name", "round"],ascending=[True, True, True]).reset_index(drop=True)

    for row in df.itertuples(index=False):
        d: pd.Timestamp = row.date
        w: str = row.winner
        l: str = row.loser
        level = row.level
        surface = row.surface

        #ESTRAGGO RATING ELO per Winner e Loser, R0 default
        Ew = R.get(w, R0)
        El = R.get(l, R0)

        pw = elo_p(Ew, El, s=S) #calcoliamo la probabilità che i batta j (ELO FORMULA) per aggiornamento ratings

        #Impostiamo decay, livello torneo e superfici, rendendoli opzionali per confronto successivo con baseline Elo standard
        if use_decay:
            age_days = max(0, (finals_date - d).days)
            decay = 2.0 ** (-age_days * inv_h)
        else:
            decay = 1.0

        lvl_w = w_level(level) if use_level_weights else 1.0
        surf_w = w_surface(surface) if use_surface_weights else 1.0

        # K effettivo per ELO-Atp finals hard surface
        k_eff = K * lvl_w * surf_w * decay

        # AGGIORNAMENTO RATING ELO DEI DUE GIOCATORI
        R[w] = Ew + k_eff * (1.0 - pw)
        R[l] = El - k_eff * (1.0 - pw)

    return R


def elo_pre_finals_for_year(matches_all: pd.DataFrame, year: int, entrants: list[str],
                            R0: float = 1500.0, K: float = 32.0, S: float = 400.0,
                            halflife_days: float = 180,
                            use_decay: bool = True,
                            use_level_weights: bool = True,
                            use_surface_weights: bool = True):

    finals_date = get_finals_start(year)

    #usiamo la funzione definita in precedenza per selezionare i match su cui volgiamo calcoalre il ranking ELO nell anno di riferimento
    train_df = get_train_matches(matches_all, year)

    # prepara df in formato atteso da train_elo
    df_elo = train_df[["tourney_date","tourney_name","round","winner_name","loser_name","tourney_level","surface"]].copy()
    df_elo = df_elo.rename(columns={"tourney_date": "date","winner_name": "winner","loser_name": "loser","tourney_level": "level"})

    rating_all = train_elo(df_elo, finals_date=finals_date, R0=R0, K=K, S=S, halflife_days=halflife_days,
                           use_decay=use_decay, use_level_weights=use_level_weights, use_surface_weights=use_surface_weights)

    rating_entrants = {p: rating_all.get(p, R0) for p in entrants}
    return rating_all, rating_entrants #dictionary completo giocatori e ranking, e ranking degli 8 qualificati

### Probability matrix

Once the pre-tournament Elo ratings of the 8 entrants have been computed, I convert them into a pairwise probability matrix \(P\) with the following function build_prob_matrix. Each entry \(P_{ij}\) represents the estimated probability that player \(i\) defeats player \(j\), obtained through the Elo logistic formula. This matrix is the probabilistic core of the simulation stage, since every subsequent match outcome is generated starting from these head-to-head probabilities.

In [10]:
# Matrice 8x8 (nxn) di P(i batte j) a partire dai rating Elo degli 8 entrants.

def build_prob_matrix(entrants: list[str],
                      ratings: dict[str, float],
                      s: float = 400.0) -> pd.DataFrame:

    n = len(entrants)
    P = np.full((n, n), np.nan, dtype=float) #creiamo la matrice nxn inizializzata a Nan

    for i in range(n):
        Ei = ratings[entrants[i]]
        for j in range(i + 1, n):
            Ej = ratings[entrants[j]]

            p_ij = elo_p(Ei, Ej, s=s)   # P(i batte j)
            P[i, j] = p_ij
            P[j, i] = 1.0 - p_ij        # Sfruttiamo la simmetria delle probabilita della matrice per riempire il triangolo inferiore

    P = pd.DataFrame(P, index=entrants, columns=entrants)

    return P

### Single tournament simulation and Monte Carlo

In this final block, I simulate the full ATP Finals format: round-robin stage, semifinals, and final. Each single match is generated by comparing a uniform random draw with the corresponding head-to-head probability in the matrix \(P\), while group rankings are constructed from the simulated number of wins, using head-to-head results for 2-player ties and Elo as a simplified tie-breaker for larger ties. Repeating the whole tournament simulation many times, I estimate the title-winning probability of each entrant as the empirical frequency with which that player wins the tournament.

In [11]:
#SCRIVIAMO UNA FUNZIONE CHE SIMULI L'ESITO DI UN SINGOLO MATCH, TRAMITE UN GENERATORE CASUALE DI NUMERI (DISTRIBUZIONE UNIFORME TRA 0 e 1) confrontata CON LA PROBABILITA CHE i BATTA j
def simulate_match(i: str, j: str, P: pd.DataFrame, rng: np.random.Generator) -> str:
    return i if rng.random() < float(P.loc[i, j]) else j #ritorna il vincitore


In [12]:
#FUNZIONE PER SIMULARE ROUND ROBIN (6 partite)

def round_robin(group: list[str], P: pd.DataFrame, rng: np.random.Generator):

    wins = Counter({p: 0 for p in group})
    h2h = {}  # per tenere conto del vincitore dello scontro diretto di ogni partita, (i,j) -> winner, nel caso di arrivo a pari vittorie di 2 giocatori

    for a in range(4):
        for b in range(a+1, 4):
            i, j = group[a], group[b]
            w = simulate_match(i, j, P, rng)
            wins[w] += 1
            h2h[frozenset([i, j])] = w

    return wins, h2h

In [13]:
#FUNZIONE PER CREARE LA CLASSIFICA DEL GRUPPO POST SIMULAZIONE GIRONE

def rank_group(group: list[str], wins: Counter, h2h: dict, ratings: dict[str, float]) -> list[str]:

    buckets = {} #dizionario che ha come key il numero di vittorie e come valore una lista con i nomi dei giocatori con quelle vittorie
    for p in group:
        buckets.setdefault(wins[p], []).append(p)

    ordered = []
    for wcount in sorted(buckets.keys(), reverse=True):
        tied = buckets[wcount] #la lista di giocatori di buckets per una certa key (numero di vittorie)

        if len(tied) == 1:
            ordered += tied #aggiungiamo a ordered il nome

        elif len(tied) == 2:
            a, b = tied
            winner = h2h[frozenset([a, b])]
            ordered += [winner, (b if winner == a else a)]

        else:
            tied = sorted(tied, key=lambda x: ratings.get(x, R0), reverse=True)
            ordered += tied

    return ordered #classifica ordinata del girone

In [14]:
#INTERA SIMULAZIONE DI UN TORNEO

def simulate_finals_once(groups, P, ratings, rng) -> str:
    # GIRONI
    winsA, h2hA = round_robin(groups["A"], P, rng)
    winsB, h2hB = round_robin(groups["B"], P, rng)

    rankA = rank_group(groups["A"], winsA, h2hA, ratings)
    rankB = rank_group(groups["B"], winsB, h2hB, ratings)

    A1, A2 = rankA[0], rankA[1]
    B1, B2 = rankB[0], rankB[1]

    # SEMIFINALI + FINALE
    sf1 = simulate_match(A1, B2, P, rng)
    sf2 = simulate_match(B1, A2, P, rng)
    champ = simulate_match(sf1, sf2, P, rng)

    return champ

In [15]:
def monte_carlo_title_probs(groups: dict[str, list[str]],
                            P: pd.DataFrame,
                            ratings: dict[str, float],
                            n_runs: int = 50_000,
                            seed: int = 0) -> pd.Series:
    rng = np.random.default_rng(seed)
    players = list(P.index)

    title_cnt = Counter({p: 0 for p in players})

    for _ in range(n_runs):
        champ = simulate_finals_once(groups, P, ratings, rng)
        title_cnt[champ] += 1

    probs = pd.Series({p: title_cnt[p] / n_runs for p in players})
    return probs.sort_values(ascending=False)

In [16]:
def run_year_once(matches_all: pd.DataFrame, year: int,
                  entrants: list[str],
                  groups: dict[str, list[str]],
                  n_runs: int = 50_000,
                  seed: int = 0,
                  use_decay: bool = True,
                  use_level_weights: bool = True,
                  use_surface_weights: bool = True) -> pd.Series:

    _, rating_entrants = elo_pre_finals_for_year(matches_all, year, entrants, R0=R0, K=K, S=S, halflife_days=HALFLIFE_DAYS,
                                                use_decay=use_decay, use_level_weights=use_level_weights, use_surface_weights=use_surface_weights)
    P = build_prob_matrix(entrants, rating_entrants, s=S)
    title_probs = monte_carlo_title_probs(groups, P, rating_entrants, n_runs=n_runs, seed=seed).astype(float)
    return title_probs

##TOY EXAMPLE
In this toy example, I apply the full resolution algorithm to the ATP Finals 2019 in order to show, on a realistic case, all the main intermediate steps of the procedure before moving to the full historical simulation.

### Ranking Elo 2019
In this block, I fix the 2019 ATP Finals entrants and groups, compute the pre-tournament Elo ratings from the selected training window, and then display both the overall Elo ranking and the ranking restricted to the 8 qualified players. The output shows the estimated strength of the entrants before the tournament and provides the starting point for the probability matrix and the subsequent tournament simulation.

In [17]:
YEAR_TOY = 2019

ENTRANTS_2019 = [
    "Rafael Nadal", "Novak Djokovic", "Roger Federer", "Daniil Medvedev",
    "Dominic Thiem", "Stefanos Tsitsipas", "Alexander Zverev", "Matteo Berrettini"
]

GROUPS_2019 = {
    "A": ["Rafael Nadal", "Daniil Medvedev", "Stefanos Tsitsipas", "Alexander Zverev"],
    "B": ["Novak Djokovic", "Roger Federer", "Dominic Thiem", "Matteo Berrettini"]
}

#Compute Elo pre-Finals 2019
finals_date_2019 = get_finals_start(YEAR_TOY)
train_df_2019 = get_train_matches(matches_all, YEAR_TOY) #estraiamo data atp 2019 e selezioniamo i match dei 12 mesi precedenti

#Rename df_elo_2019 columns as expected by train_elo
df_elo_2019 = train_df_2019[["tourney_date","tourney_name", "round", "winner_name", "loser_name", "tourney_level", "surface"]].copy()
df_elo_2019 = df_elo_2019.rename(columns={"tourney_date": "date","winner_name": "winner","loser_name": "loser","tourney_level": "level"})


rating_all_2019 = train_elo(
    df_elo_2019,
    finals_date=finals_date_2019,
    R0=R0, K=K, S=S, halflife_days=HALFLIFE_DAYS
)

rating_entrants_2019 = {p: rating_all_2019.get(p, R0) for p in ENTRANTS_2019}

# Full rating leaderboard (top 20) + entrants-only
leaderboard_all = pd.Series(rating_all_2019).sort_values(ascending=False)
leaderboard_entrants = pd.Series(rating_entrants_2019).sort_values(ascending=False)

print("\nRanking Elo of the first 20:")
display(leaderboard_all.head(20).to_frame("Elo"))

print("\nElo of the 8 entrants (sorted):")
display(leaderboard_entrants.to_frame("Elo"))



Ranking Elo of the first 20:


,Elo
Rafael Nadal,1807.110086
Novak Djokovic,1790.427926
Daniil Medvedev,1775.214404
Roger Federer,1762.347490
Dominic Thiem,1695.961040
Denis Shapovalov,1675.573047
Stefanos Tsitsipas,1668.032841
Matteo Berrettini,1647.575781
Andrey Rublev,1645.984810
Alex De Minaur,1643.885780



Elo of the 8 entrants (sorted):


,Elo
Rafael Nadal,1807.110086
Novak Djokovic,1790.427926
Daniil Medvedev,1775.214404
Roger Federer,1762.347490
Dominic Thiem,1695.961040
Stefanos Tsitsipas,1668.032841
Matteo Berrettini,1647.575781
Alexander Zverev,1633.466498


In [18]:
#Per mostrare un mini aggiornamento della classifica ELO (nel report), che costituisce un micro step dell algoritmo ELO, salviamo i primi tre match della finestra di training del 2019
demo_matches = df_elo_2019.sort_values(["date", "tourney_name", "round"]).head(3).copy()
display(demo_matches)

,date,tourney_name,round,winner,loser,level,surface
58915,2018-11-23,Davis Cup WG F: FRA vs CRO,NaN,Borna Coric,Jeremy Chardy,D,Clay
58916,2018-11-23,Davis Cup WG F: FRA vs CRO,NaN,Marin Cilic,Jo-Wilfried Tsonga,D,Clay
58917,2018-11-23,Davis Cup WG F: FRA vs CRO,NaN,Marin Cilic,Lucas Pouille,D,Clay


###Probability matrix P_2019
In this step, I build the head-to-head probability matrix \(P_2019\) for the 8 entrants of the 2019 ATP Finals.

In [19]:
#Matrice di probabilità per gli 8 qualificati

P_2019 = build_prob_matrix(ENTRANTS_2019, rating_entrants_2019, s=S)

display(P_2019.round(3))

,Rafael Nadal,Novak Djokovic,Roger Federer,Daniil Medvedev,Dominic Thiem,Stefanos Tsitsipas,Alexander Zverev,Matteo Berrettini
Rafael Nadal,NaN,0.524,0.564,0.546,0.655,0.690,0.731,0.715
Novak Djokovic,0.476,NaN,0.540,0.522,0.633,0.669,0.712,0.695
Roger Federer,0.436,0.460,NaN,0.481,0.594,0.632,0.677,0.659
Daniil Medvedev,0.454,0.478,0.519,NaN,0.612,0.650,0.693,0.676
Dominic Thiem,0.345,0.367,0.406,0.388,NaN,0.540,0.589,0.569
Stefanos Tsitsipas,0.310,0.331,0.368,0.350,0.460,NaN,0.550,0.529
Alexander Zverev,0.269,0.288,0.323,0.307,0.411,0.450,NaN,0.480
Matteo Berrettini,0.285,0.305,0.341,0.324,0.431,0.471,0.520,NaN


###Single tournament simulation
Now I simulate one full edition of the 2019 ATP Finals, starting from the round-robin stage in both groups and then moving to semifinals and final. The outputs show the simulated number of wins, the head-to-head results used for tie-breaking, the final group rankings, the qualified players, and the champion of this single tournament run.

In [20]:
#One full tournament simulation
rng_demo = np.random.default_rng(2019)

#simulation Group A round robin
winsA, h2hA = round_robin(GROUPS_2019["A"], P_2019, rng_demo)
print("\nGroup A - wins:")
print(dict(winsA))
print("Group A - h2h ( pair -> winner):")
print({tuple(sorted(list(k))): v for k, v in h2hA.items()})

rankA = rank_group(GROUPS_2019["A"], winsA, h2hA, rating_entrants_2019)
print("\nGroup A - final ranking (after tie-break):")
print(rankA)

A1, A2 = rankA[0], rankA[1]
print("Qualified from Group A:", "A1 =", A1, ", A2 =", A2)

# simulation Group B round robin
winsB, h2hB = round_robin(GROUPS_2019["B"], P_2019, rng_demo)
print("\nGroup B - wins:")
print(dict(winsB))
print("Group B - h2h (pair -> winner):")
print({tuple(sorted(list(k))): v for k, v in h2hB.items()})

rankB = rank_group(GROUPS_2019["B"], winsB, h2hB, rating_entrants_2019)
print("\nGroup B - final ranking (after tie-break):")
print(rankB)

B1, B2 = rankB[0], rankB[1]
print("Qualified from Group B:", "B1 =", B1, ", B2 =", B2)

#  Semifinals and Final
print("\nSemifinals:")
print("SF1:", A1, "vs", B2, "| P(A1 wins) =", float(P_2019.loc[A1, B2]))
sf1_winner = simulate_match(A1, B2, P_2019, rng_demo)
print(" -> SF1 winner:", sf1_winner)

print("SF2:", B1, "vs", A2, "| P(B1 wins) =", float(P_2019.loc[B1, A2]))
sf2_winner = simulate_match(B1, A2, P_2019, rng_demo)
print(" -> SF2 winner:", sf2_winner)

print("\nFinal:")
print("F:", sf1_winner, "vs", sf2_winner, "| P(sf1 wins) =", float(P_2019.loc[sf1_winner, sf2_winner]))
champ_demo = simulate_match(sf1_winner, sf2_winner, P_2019, rng_demo)
print(" -> Champion:", champ_demo)


Group A - wins:
{'Rafael Nadal': 3, 'Daniil Medvedev': 1, 'Stefanos Tsitsipas': 2, 'Alexander Zverev': 0}
Group A - h2h ( pair -> winner):
{('Daniil Medvedev', 'Rafael Nadal'): 'Rafael Nadal', ('Rafael Nadal', 'Stefanos Tsitsipas'): 'Rafael Nadal', ('Alexander Zverev', 'Rafael Nadal'): 'Rafael Nadal', ('Daniil Medvedev', 'Stefanos Tsitsipas'): 'Stefanos Tsitsipas', ('Alexander Zverev', 'Daniil Medvedev'): 'Daniil Medvedev', ('Alexander Zverev', 'Stefanos Tsitsipas'): 'Stefanos Tsitsipas'}

Group A - final ranking (after tie-break):
['Rafael Nadal', 'Stefanos Tsitsipas', 'Daniil Medvedev', 'Alexander Zverev']
Qualified from Group A: A1 = Rafael Nadal , A2 = Stefanos Tsitsipas

Group B - wins:
{'Novak Djokovic': 2, 'Roger Federer': 1, 'Dominic Thiem': 1, 'Matteo Berrettini': 2}
Group B - h2h (pair -> winner):
{('Novak Djokovic', 'Roger Federer'): 'Novak Djokovic', ('Dominic Thiem', 'Novak Djokovic'): 'Dominic Thiem', ('Matteo Berrettini', 'Novak Djokovic'): 'Novak Djokovic', ('Dominic T

###Monte Carlo Simulation
Finally, I repeat the full tournament simulation 10,000 times and estimate the title-winning probability of each entrant as the relative frequency with which that player wins the tournament.

In [21]:
# Monte Carlo
print(" Monte Carlo estimation:")
probs_2019 = monte_carlo_title_probs(
    groups=GROUPS_2019,
    P=P_2019,
    ratings=rating_entrants_2019,
    n_runs=10_000,
    seed=0
)

display((probs_2019 * 100).round(2).to_frame("Title probability (%)"))

 Monte Carlo estimation:


,Title probability (%)
Rafael Nadal,26.84
Novak Djokovic,22.59
Daniil Medvedev,19.08
Roger Federer,16.87
Dominic Thiem,5.87
Stefanos Tsitsipas,3.95
Matteo Berrettini,2.62
Alexander Zverev,2.18


## FULL SIMULATION

In this final section, I run the complete historical backtesting of the algorithm on all ATP Finals editions from 2000 to 2024. For each year, I use the official entrants and group composition, estimate the pre-tournament title probabilities, and then compare them with the real tournament outcome through ranking-based and probabilistic metrics. Finally, I compare the full weighted Elo model with a baseline version of standard Elo in order to evaluate whether the additional adjustments improve predictive performance.


In [22]:
# Entrants: dizionario dgli otto qualificati con chiave l'anno
# Groups: A e B corrispondono al "primo gruppo elencato" e "secondo gruppo elencato" presi dal sito ufficiale delle Atp Finals. La previsione delle probabilita di vincere avviene generalmente dopo il sorteggio dei gironi, dopo cui i siti di scommesse pubblicano le quote antepost

groups_by_year = {
    2000: {"A": ["Pete Sampras", "Marat Safin", "Lleyton Hewitt", "Alex Corretja"],
           "B": ["Andre Agassi", "Gustavo Kuerten", "Yevgeny Kafelnikov", "Magnus Norman"]},

    2001: {"A": ["Gustavo Kuerten", "Juan Carlos Ferrero", "Yevgeny Kafelnikov", "Goran Ivanisevic"],
           "B": ["Lleyton Hewitt", "Andre Agassi", "Patrick Rafter", "Sebastien Grosjean"]},

    2002: {"A": ["Carlos Moya", "Lleyton Hewitt", "Albert Costa", "Marat Safin"],
           "B": ["Roger Federer", "Juan Carlos Ferrero", "Jiri Novak", "Andre Agassi"]},

    2003: {"A": ["Andy Roddick", "Guillermo Coria", "Rainer Schuettler", "Carlos Moya"],
           "B": ["Juan Carlos Ferrero", "Roger Federer", "Andre Agassi", "David Nalbandian"]},

    2004: {"A": ["Roger Federer", "Lleyton Hewitt", "Carlos Moya", "Gaston Gaudio"],
           "B": ["Andy Roddick", "Marat Safin", "Guillermo Coria", "Tim Henman"]},

    2005: {"A": ["Roger Federer", "Guillermo Coria", "Ivan Ljubicic", "David Nalbandian"],
           "B": ["Andre Agassi", "Nikolay Davydenko", "Gaston Gaudio", "Mariano Puerta"]},

    2006: {"A": ["Roger Federer", "Ivan Ljubicic", "Andy Roddick", "David Nalbandian"],
           "B": ["Rafael Nadal", "Nikolay Davydenko", "Tommy Robredo", "James Blake"]},

    2007: {"A": ["Roger Federer", "Nikolay Davydenko", "Andy Roddick", "Fernando Gonzalez"],
           "B": ["Rafael Nadal", "Novak Djokovic", "David Ferrer", "Richard Gasquet"]},

    2008: {"A": ["Roger Federer", "Andy Murray", "Andy Roddick", "Gilles Simon"],
           "B": ["Novak Djokovic", "Nikolay Davydenko", "Jo-Wilfried Tsonga", "Juan Martin del Potro"]},

    2009: {"A": ["Roger Federer", "Andy Murray", "Juan Martin del Potro", "Fernando Verdasco"],
           "B": ["Rafael Nadal", "Novak Djokovic", "Nikolay Davydenko", "Robin Soderling"]},

    2010: {"A": ["Rafael Nadal", "Novak Djokovic", "Tomas Berdych", "Andy Roddick"],
           "B": ["Roger Federer", "Robin Soderling", "Andy Murray", "David Ferrer"]},

    2011: {"A": ["Novak Djokovic", "Andy Murray", "David Ferrer", "Tomas Berdych"],
           "B": ["Rafael Nadal", "Roger Federer", "Jo-Wilfried Tsonga", "Mardy Fish"]},

    2012: {"A": ["Novak Djokovic", "Andy Murray", "Tomas Berdych", "Jo-Wilfried Tsonga"],
           "B": ["Roger Federer", "David Ferrer", "Juan Martin del Potro", "Janko Tipsarevic"]},

    2013: {"A": ["Rafael Nadal", "Stanislas Wawrinka", "Tomas Berdych", "David Ferrer"],
           "B": ["Novak Djokovic", "Roger Federer", "Juan Martin del Potro", "Richard Gasquet"]},

    2014: {"A": ["Novak Djokovic", "Stan Wawrinka", "Tomas Berdych", "Marin Cilic"],
           "B": ["Roger Federer", "Kei Nishikori", "Andy Murray", "Milos Raonic"]},

    2015: {"A": ["Novak Djokovic", "Roger Federer", "Tomas Berdych", "Kei Nishikori"],
           "B": ["Andy Murray", "Stan Wawrinka", "Rafael Nadal", "David Ferrer"]},

    2016: {"A": ["Andy Murray", "Stan Wawrinka", "Kei Nishikori", "Marin Cilic"],
           "B": ["Novak Djokovic", "Milos Raonic", "Gael Monfils", "Dominic Thiem"]},

    2017: {"A": ["Rafael Nadal", "Dominic Thiem", "Grigor Dimitrov", "David Goffin"],
           "B": ["Roger Federer", "Alexander Zverev", "Marin Cilic", "Jack Sock"]},

    2018: {"A": ["Roger Federer", "Kevin Anderson", "Dominic Thiem", "Kei Nishikori"],
           "B": ["Novak Djokovic", "Alexander Zverev", "Marin Cilic", "John Isner"]},

    2019: {"A": ["Rafael Nadal", "Daniil Medvedev", "Stefanos Tsitsipas", "Alexander Zverev"],
           "B": ["Novak Djokovic", "Roger Federer", "Dominic Thiem", "Matteo Berrettini"]},

    2020: {"A": ["Novak Djokovic", "Daniil Medvedev", "Alexander Zverev", "Diego Schwartzman"],
           "B": ["Rafael Nadal", "Dominic Thiem", "Stefanos Tsitsipas", "Andrey Rublev"]},

    2021: {"A": ["Novak Djokovic", "Stefanos Tsitsipas", "Andrey Rublev", "Casper Ruud"],
           "B": ["Daniil Medvedev", "Alexander Zverev", "Matteo Berrettini", "Hubert Hurkacz"]},

    2022: {"A": ["Rafael Nadal", "Casper Ruud", "Felix Auger-Aliassime", "Taylor Fritz"],
           "B": ["Stefanos Tsitsipas", "Daniil Medvedev", "Andrey Rublev", "Novak Djokovic"]},

    2023: {"A": ["Novak Djokovic", "Jannik Sinner", "Stefanos Tsitsipas", "Holger Rune"],
           "B": ["Carlos Alcaraz", "Daniil Medvedev", "Andrey Rublev", "Alexander Zverev"]},

    2024: {"A": ["Jannik Sinner", "Daniil Medvedev", "Taylor Fritz", "Alex de Minaur"],
           "B": ["Alexander Zverev", "Carlos Alcaraz", "Casper Ruud", "Andrey Rublev"]},

}

entrants_by_year = {year: groups_by_year[year]["A"] + groups_by_year[year]["B"] for year in groups_by_year}

In [23]:
#Funzione per estrarre il vincitore reale di ogni torneo delle finals sfruttando la funzione definita in precedenza che seleziona i match delle finals
def real_finals_champion(matches_all: pd.DataFrame, year: int) -> str:
    fm = get_finals_matches(matches_all, year)
    final = fm[fm["round"] == "F"]
    return str(final.iloc[0]["winner_name"])


def tournament_metrics_from_probs(probs: pd.Series, champ_real: str) -> dict:

    p_champ = float(probs[champ_real]) #PROBABILITY OF THE REAL CHAMPION
    rank_champ = int(probs.index.get_loc(champ_real) + 1) #RANK OF THE REAL CHAMPION
    logloss = float(-np.log(p_champ)) #LOG-LOSS

    return {
        "p_champ": p_champ,
        "rank_champ": rank_champ,
        "top1": int(rank_champ <= 1),
        "top2": int(rank_champ <= 2),
        "top3": int(rank_champ <= 3),
        "logloss": logloss,
    }


def backtest_years(matches_all: pd.DataFrame,
                   years: list[int],
                   n_runs: int = 50_000,
                   seed: int = 0,
                   use_decay: bool = True,
                   use_level_weights: bool = True,
                   use_surface_weights: bool = True) -> pd.DataFrame:
    rows = []
    #per ogni anno dal 2000 al 2024 run the full prediction pipeline, and store the evaluation metrics in rows
    for y in years:
        champ = real_finals_champion(matches_all, y)
        probs = run_year_once(matches_all, y,
                              entrants_by_year[y],
                              groups_by_year[y],
                              n_runs=n_runs,
                              seed=seed,
                              use_decay=use_decay,
                              use_level_weights=use_level_weights,
                              use_surface_weights=use_surface_weights)

        rows.append({"year": y, "champ_real": champ, **tournament_metrics_from_probs(probs, champ)})

    out = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)

    if not out.empty:
        summary = {
            "top1_acc": float(out["top1"].mean()),
            "top2_hit": float(out["top2"].mean()),
            "top3_hit": float(out["top3"].mean()),
            "mean_logloss": float(out["logloss"].mean()),
        }
        print("Backtest summary in all years:", summary)

    return out

In [24]:
YEARS_BACKTEST = sorted(groups_by_year.keys())
bt_full = backtest_years(matches_all, YEARS_BACKTEST, n_runs=20000, seed=0)
display(bt_full)


Backtest summary in all years: {'top1_acc': 0.52, 'top2_hit': 0.64, 'top3_hit': 0.68, 'mean_logloss': 1.585475001740103}


,year,champ_real,p_champ,rank_champ,top1,top2,top3,logloss
0,2000,Gustavo Kuerten,0.09620,5,0,0,0,2.341326
1,2001,Lleyton Hewitt,0.30410,1,1,1,1,1.190399
2,2002,Lleyton Hewitt,0.24220,2,0,1,1,1.417991
3,2003,Roger Federer,0.12760,3,0,0,1,2.058855
4,2004,Roger Federer,0.38790,1,1,1,1,0.947008
5,2005,David Nalbandian,0.03930,5,0,0,0,3.236531
6,2006,Roger Federer,0.56805,1,1,1,1,0.565546
7,2007,Roger Federer,0.41300,1,1,1,1,0.884308
8,2008,Novak Djokovic,0.11280,4,0,0,0,2.182139
9,2009,Nikolay Davydenko,0.05490,6,0,0,0,2.902242


In [25]:
bt_baseline = backtest_years(matches_all, YEARS_BACKTEST, n_runs=20000, seed=0,
                             use_decay=False,
                             use_level_weights=False,
                             use_surface_weights=False)

display(bt_baseline)

Backtest summary in all years: {'top1_acc': 0.52, 'top2_hit': 0.6, 'top3_hit': 0.72, 'mean_logloss': 1.6137312337832714}


,year,champ_real,p_champ,rank_champ,top1,top2,top3,logloss
0,2000,Gustavo Kuerten,0.12130,3,0,0,1,2.109488
1,2001,Lleyton Hewitt,0.32365,1,1,1,1,1.128093
2,2002,Lleyton Hewitt,0.24555,2,0,1,1,1.404255
3,2003,Roger Federer,0.13950,4,0,0,0,1.969691
4,2004,Roger Federer,0.45915,1,1,1,1,0.778378
5,2005,David Nalbandian,0.02385,5,0,0,0,3.735971
6,2006,Roger Federer,0.62840,1,1,1,1,0.464578
7,2007,Roger Federer,0.42975,1,1,1,1,0.844552
8,2008,Novak Djokovic,0.15205,3,0,0,1,1.883546
9,2009,Nikolay Davydenko,0.04170,6,0,0,0,3.177254


In [26]:
#Confronto tra ELO standard (base) e ELO weighted (full)
print("FULL:", {
    "top1": float(bt_full["top1"].mean()),
    "top2": float(bt_full["top2"].mean()),
    "top3": float(bt_full["top3"].mean()),
    "logloss": float(bt_full["logloss"].mean())
})
print("BASE:", {
    "top1": float(bt_baseline["top1"].mean()),
    "top2": float(bt_baseline["top2"].mean()),
    "top3": float(bt_baseline["top3"].mean()),
    "logloss": float(bt_baseline["logloss"].mean())
})

FULL: {'top1': 0.52, 'top2': 0.64, 'top3': 0.68, 'logloss': 1.585475001740103}
BASE: {'top1': 0.52, 'top2': 0.6, 'top3': 0.72, 'logloss': 1.6137312337832714}
